# Predict a New NYC Taxi Trip

Use this notebook when you want to pass a new taxi trip into the trained model. The model predicts whether the trip is likely to last **30 minutes or longer**.

## What This Model Predicts

The model predicts the binary target `long_trip`:

- `1`: Long trip, 30 minutes or more
- `0`: Short trip, under 30 minutes

It does **not** predict exact trip duration. It predicts the class and probability of a 30+ minute trip.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

repo_url = "https://github.com/soumithkumara/project_day_1.git"
repo_dir = Path("/content/project_day_1")

# If the notebook was opened directly from GitHub, clone the full repo first.
if not Path("requirements.txt").exists():
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
    os.chdir(repo_dir)

# Pull the latest changes if this is already a cloned repo.
if Path(".git").exists():
    subprocess.run(["git", "pull", "origin", "main"], check=True)

print(f"Working directory: {Path.cwd()}")

## Install Requirements

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

## Train or Load the Model

The trained model file is ignored by Git because it is generated output. If Colab does not already have it, this cell runs the main pipeline once to create it.

In [ ]:
model_path = Path("models/long_trip_logistic_model.joblib")

if not model_path.exists():
    print("Model file not found. Running the training pipeline first...")
    subprocess.run([sys.executable, "src/nyc_taxi_long_trip_model.py"], check=True)
else:
    print(f"Using existing model: {model_path}")

print(f"Model ready: {model_path.exists()}")

## Prediction Function

The model needs the same feature columns it was trained with. This function takes simple trip details, creates the model features, and returns the predicted class plus probability.

In [ ]:
import joblib
import pandas as pd

model = joblib.load(model_path)

# TLC location IDs for Newark, JFK, and LaGuardia airport zones.
AIRPORT_ZONE_IDS = {1, 132, 138}

def build_trip_features(
    pickup_datetime,
    trip_distance,
    passenger_count,
    vendor_id,
    ratecode_id,
    pu_location_id,
    do_location_id,
):
    pickup_datetime = pd.to_datetime(pickup_datetime)

    return pd.DataFrame([{
        "trip_distance": float(trip_distance),
        "passenger_count": int(passenger_count),
        "pickup_hour": pickup_datetime.hour,
        "pickup_dayofweek": pickup_datetime.dayofweek,
        "is_weekend": int(pickup_datetime.dayofweek in [5, 6]),
        "airport_trip": int(
            int(pu_location_id) in AIRPORT_ZONE_IDS
            or int(do_location_id) in AIRPORT_ZONE_IDS
        ),
        "VendorID": str(vendor_id),
        "RatecodeID": str(ratecode_id),
        "PULocationID": str(pu_location_id),
        "DOLocationID": str(do_location_id),
    }])

def predict_long_trip(
    pickup_datetime,
    trip_distance,
    passenger_count,
    vendor_id,
    ratecode_id,
    pu_location_id,
    do_location_id,
):
    features = build_trip_features(
        pickup_datetime=pickup_datetime,
        trip_distance=trip_distance,
        passenger_count=passenger_count,
        vendor_id=vendor_id,
        ratecode_id=ratecode_id,
        pu_location_id=pu_location_id,
        do_location_id=do_location_id,
    )

    prediction = int(model.predict(features)[0])
    probability = float(model.predict_proba(features)[0][1])

    return features, pd.DataFrame([{
        "prediction_label": "Long trip: 30 minutes or more" if prediction == 1 else "Short trip: under 30 minutes",
        "predicted_long_trip_class": prediction,
        "long_trip_probability": round(probability, 4),
        "long_trip_probability_percent": f"{probability:.2%}",
    }])

## Enter One New Taxi Trip

Change the values below and run the cell. For pickup and dropoff zones, use NYC TLC taxi zone IDs. Common airport zone IDs are `132` for JFK, `138` for LaGuardia, and `1` for Newark.

In [ ]:
# @title New Taxi Trip Input
pickup_datetime = "2024-01-15 17:30:00"  # @param {type:"string"}
trip_distance = 14.2  # @param {type:"number"}
passenger_count = 2  # @param {type:"integer"}
vendor_id = 2  # @param {type:"integer"}
ratecode_id = 1  # @param {type:"integer"}
pu_location_id = 132  # @param {type:"integer"}
do_location_id = 48  # @param {type:"integer"}

features, prediction_result = predict_long_trip(
    pickup_datetime=pickup_datetime,
    trip_distance=trip_distance,
    passenger_count=passenger_count,
    vendor_id=vendor_id,
    ratecode_id=ratecode_id,
    pu_location_id=pu_location_id,
    do_location_id=do_location_id,
)

print("Model input features:")
display(features)

print("Prediction:")
display(prediction_result)

## Predict Multiple Trips at Once

Optional: edit the rows below if you want to test several taxi trips in one run.

In [ ]:
new_trips = [
    {
        "pickup_datetime": "2024-01-15 17:30:00",
        "trip_distance": 14.2,
        "passenger_count": 2,
        "vendor_id": 2,
        "ratecode_id": 1,
        "pu_location_id": 132,
        "do_location_id": 48,
    },
    {
        "pickup_datetime": "2024-01-16 10:15:00",
        "trip_distance": 1.4,
        "passenger_count": 1,
        "vendor_id": 2,
        "ratecode_id": 1,
        "pu_location_id": 237,
        "do_location_id": 236,
    },
]

batch_results = []
for i, trip in enumerate(new_trips, start=1):
    _, result = predict_long_trip(**trip)
    row = {"trip_number": i, **trip, **result.iloc[0].to_dict()}
    batch_results.append(row)

display(pd.DataFrame(batch_results))